In [21]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage
from dotenv import load_dotenv, find_dotenv
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
import os
from langgraph.graph.message import add_messages 

In [22]:
load_dotenv(find_dotenv())

True

In [23]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_TOKEN")
)

In [24]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [25]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = model.invoke(messages)
    return {'messages': [response]}

In [26]:
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile()

In [27]:
while True:
    user_message = input("Type here: ")

    print("User: ", user_message)

    if user_message.strip().lower() in ['exit', 'bye', 'quit']:
        break

    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]})

    print("AI: ", response['messages'][-1].content)

User:  hey
AI:  Hello! How can I assist you today?
User:  bye
